# 🔤 Assamese OCR — Sentence Model Training

This notebook trains a CRNN (CNN + LSTM) model for Assamese sentence-level OCR using CTC loss.

**Before you start:**
1. Set runtime to **GPU** → Runtime > Change runtime type > T4 GPU
2. Upload your checkpoints to Google Drive (see Setup section)

---

## 1. Setup — Mount Drive & Clone Repo

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone your training repo (replace with your actual repo URL)
# If already cloned, skip this cell
REPO_URL = "https://github.com/YOUR_USERNAME/assamese-ocr-training.git"  # ← CHANGE THIS
REPO_NAME = "assamese-ocr-training"  # ← match your repo name
BRANCH = "develop"  # ← cloning the develop branch

import os
if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone -b {BRANCH} {REPO_URL}
else:
    print(f'Repo already exists at /content/{REPO_NAME}')
    # Pull latest changes from develop
    !cd /content/{REPO_NAME} && git checkout {BRANCH} && git pull

In [ ]:
# Set working directory
WORK_DIR = f'/content/{REPO_NAME}/django_backend'
%cd {WORK_DIR}
!pwd

In [ ]:
# Install dependencies
!pip install -q -r requirements-train.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Link Drive Assets (Checkpoints & Data)

**First time only:** Create this folder structure in your Google Drive:
```
My Drive/
└── assamese_ocr_assets/
    ├── checkpoints/
    │   ├── best_model_sentences.pth
    │   └── best_model_fast.pth  (optional, for transfer learning)
    └── data/  (will be created by the split builder)
```

Upload your checkpoint files from `django_backend/checkpoints/` into that Drive folder.

In [ ]:
# Link Drive folders to expected local paths
DRIVE_ASSETS = '/content/drive/MyDrive/assamese_ocr_assets'

# Create Drive directories if they don't exist
os.makedirs(f'{DRIVE_ASSETS}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ASSETS}/data', exist_ok=True)

# Remove existing dirs/symlinks if present
!rm -rf checkpoints 2>/dev/null; ln -s {DRIVE_ASSETS}/checkpoints checkpoints

# For data: keep the corpus in the repo, but symlink generated splits from Drive
# so they persist across sessions.
# We only symlink the generated split dirs, not the whole data/ folder,
# because data/as-wiki-2021.txt is already in the repo.
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    src = f'{DRIVE_ASSETS}/data/{split}'
    dst = f'data/{split}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst) or os.path.exists(dst):
        !rm -rf {dst}
    os.symlink(src, dst)

print('Symlinks created:')
!ls -la checkpoints
!ls -la data/

## 3. Generate Non-Overlapping Train/Val Splits

> Skip this if you've already generated splits in a previous session (they persist on Drive).

In [ ]:
# Check if splits already exist on Drive
train_labels = f'{DRIVE_ASSETS}/data/train_real_sentences/labels/labels.txt'
if os.path.exists(train_labels):
    with open(train_labels) as f:
        n = sum(1 for _ in f)
    print(f'✅ Splits already exist on Drive! ({n} training samples)')
    print('Skip the next cell unless you want to regenerate.')
else:
    print('❌ No splits found. Run the next cell to generate them.')

In [ ]:
# Generate clean, non-overlapping splits from the wiki corpus
# This takes ~5-15 min depending on counts
!python build_real_sentence_splits.py \
    --input data/as-wiki-2021.txt \
    --train-output data/train_real_sentences \
    --val-output data/val_real_sentences \
    --test-output data/test_real_sentences \
    --train-count 15000 \
    --val-count 4000 \
    --test-count 1000 \
    --seed 42

## 4. Train the Sentence Model 🚀

In [ ]:
# Main training run
!python train_sentence_model.py \
    --train-img-dir data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir data/val_real_sentences/images \
    --val-label-file data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --plot-out training_curve_sentences.png

In [ ]:
# View training curve
from IPython.display import Image as IPImage, display
if os.path.exists('training_curve_sentences.png'):
    display(IPImage('training_curve_sentences.png'))
else:
    print('No training curve found yet.')

## 5. Test Predictions

In [ ]:
# Quick prediction on a test image
!python predict_cli.py \
    --image data/test_real_sentences/images/sentence_000000.png \
    --checkpoint checkpoints/best_model_sentences.pth

In [ ]:
# Batch predict on multiple test images
import glob
test_images = sorted(glob.glob('data/test_real_sentences/images/*.png'))[:5]

for img_path in test_images:
    print(f'\n--- {os.path.basename(img_path)} ---')
    !python predict_cli.py --image {img_path} --checkpoint checkpoints/best_model_sentences.pth

## 6. (Optional) Transfer Learning

If you have `best_model_fast.pth` in `checkpoints/`, you can fine-tune from the character model.

In [ ]:
# Transfer learning (uses model_old.py architecture + best_model_fast.pth)
# !python train_transfer_learning.py

## 7. Save & Sync

Checkpoints auto-save to Drive via the symlink. To push code changes back:

In [ ]:
# Check checkpoint sizes on Drive
!ls -lh checkpoints/
print('\n✅ These are already on Drive — they persist across sessions.')

In [ ]:
# Push code changes back to GitHub (if you modified any .py files)
# %cd /content/{REPO_NAME}
# !git add -A
# !git commit -m "Colab training updates"
# !git push origin develop